# PiShield — Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/main/examples/general_usage/PiShield_quickstart.ipynb)

This single notebook bundles **all three general-usage examples** and runs **end-to-end with
no external downloads** (the requirements files are written inline):

1. **Inference** — correct a network's predictions with a Shield Layer.
2. **Training** — train a model with a Shield Layer vs. an unconstrained baseline.
3. **Shield Loss** — *encourage* requirement satisfaction with a t-norm based loss.

On Google Colab, **run the setup cell below first** to install PiShield; then run the rest
top to bottom.

## Setup

Installs PiShield if it is not already available (e.g. on a fresh Colab runtime). Running
this locally with PiShield already installed is a no-op.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("pishield") is None:
    print("Installing PiShield ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/mihaela-stoian/PiShield.git@main"],
        check=True,
    )

import torch
print("PiShield is ready. torch", torch.__version__)

---
## 1. Inference — correcting predictions with a Shield Layer

A *Shield Layer* corrects the outputs of a network so that they are **guaranteed** to satisfy
a set of requirements. A requirements file starts with an `ordering` line listing the
variables, followed by one linear inequality per line. Here we require the 3-dimensional
output to be **non-increasing**: `y_0 >= y_1 >= y_2`.

In [ ]:
from pishield.shield_layer import build_shield_layer

requirements_path = "monotonic_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("ordering y_0 y_1 y_2\n")  # order in which variables are corrected
    f.write("y_0 - y_1 >= 0\n")        # y_0 >= y_1
    f.write("y_1 - y_2 >= 0\n")        # y_1 >= y_2

print(open(requirements_path).read())

Build the Shield Layer (the requirement *type* — linear here — is auto-detected) and apply
it to some deliberately-invalid predictions.

In [ ]:
shield_layer = build_shield_layer(num_variables=3, requirements_filepath=requirements_path)

predictions = torch.tensor([
    [0.2, 0.9, 0.5],   # not monotone: 0.2 < 0.9
    [1.0, 0.0, 0.3],   # not monotone: 0.0 < 0.3
])
corrected = shield_layer(predictions.clone())

print("original  predictions:\n", predictions)
print("\ncorrected predictions:\n", corrected)

In [ ]:
def is_monotone_non_increasing(t, tol=1e-6):
    return bool((t[:, 0] >= t[:, 1] - tol).all() and (t[:, 1] >= t[:, 2] - tol).all())

print("original  satisfy requirements?", is_monotone_non_increasing(predictions))
print("corrected satisfy requirements?", is_monotone_non_increasing(corrected))

---
## 2. Training — using a Shield Layer in a model

The Shield Layer is **differentiable**, so we can embed it in a network and train through it.
We train a tiny MLP whose output must satisfy the same `y_0 >= y_1 >= y_2` requirement, and
compare it against an unconstrained baseline. We use non-increasing targets with **small
gaps**, so the ordering is easy to get wrong — an unconstrained model will invert some of
them, which is exactly what the requirement forbids.

In [ ]:
import torch.nn as nn
torch.manual_seed(0)

N, in_dim, out_dim = 512, 4, 3
X = torch.randn(N, in_dim)
base = torch.rand(N, 1)
gaps = torch.rand(N, 2) * 0.05  # small gaps -> ordering is non-trivial to preserve
Y = torch.cat([base, base - gaps[:, :1], base - gaps.sum(1, keepdim=True)], dim=1)
print("example target (non-increasing):", [round(v, 2) for v in Y[0].tolist()])

`ShieldedMLP` applies the Shield Layer to the MLP's output inside `forward` — that is the
*only* difference from the plain baseline.

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, out_dim))

class ShieldedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = make_mlp()
        self.shield = build_shield_layer(out_dim, requirements_path)

    def forward(self, x):
        return self.shield(self.model(x))

In [ ]:
def num_violations(out, tol=1e-6):
    bad = (out[:, 0] < out[:, 1] - tol) | (out[:, 1] < out[:, 2] - tol)
    return int(bad.sum())

def train(model, epochs=300, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
    return loss.item()

baseline, shielded = make_mlp(), ShieldedMLP()
base_mse, shield_mse = train(baseline), train(shielded)

with torch.no_grad():
    base_out, shield_out = baseline(X), shielded(X)

print(f"baseline : final MSE={base_mse:.4f}, violations={num_violations(base_out)}/{N}")
print(f"shielded : final MSE={shield_mse:.4f}, violations={num_violations(shield_out)}/{N}")

The **shielded** model has **zero** violations by construction, while still learning the task
(the unconstrained baseline leaves many outputs violating the requirement).

---
## 3. Shield Loss — encouraging satisfaction during training

The *Shield Loss* is an alternative for **propositional** requirements. Instead of correcting
the outputs, it adds a differentiable penalty (via a t-norm) that *encourages* the network to
satisfy the requirements — it does not *guarantee* them, but it can be added to any task loss.

Requirements are read in the Horn-rule format `<id> <head> :- <body>`, where literals are
variable indices and an `n` prefix denotes negation. Our requirement is the implication
`0 -> 1` ("if variable 0 is true then variable 1 must be true"), i.e. head `1`, body `0`.

In [ ]:
from pishield.shield_loss import build_shield_loss

requirements_path = "implication_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("c0 1 :- 0\n")   # variable 0 implies variable 1

shield_loss = build_shield_loss(num_variables=2, requirements_filepath=requirements_path,
                                tnorm_choice="product")

We start from logits that **violate** the rule (high `P(0)`, low `P(1)`) and minimise the
Shield Loss. The loss takes *probabilities* in `[0, 1]`, so we pass `sigmoid(logits)`.

In [ ]:
logits = torch.tensor([[3.0, -3.0]], requires_grad=True)  # P(0)~0.95, P(1)~0.05 -> violates 0 -> 1
opt = torch.optim.SGD([logits], lr=1.0)

for step in range(300):
    probs = torch.sigmoid(logits)
    loss = shield_loss(probs)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 60 == 0:
        p = [round(v, 2) for v in probs.detach().squeeze().tolist()]
        print(f"step {step:3d}  loss={loss.item():.4f}  probs={p}")

print("\nfinal probs:", [round(v, 2) for v in torch.sigmoid(logits).detach().squeeze().tolist()])

The loss decreases and the probabilities move to satisfy `0 -> 1`. In a real model you would
add it to your task loss, e.g. `total_loss = task_loss + shield_loss(probs)`.

---
**Recap.** Use a **Shield Layer** (parts 1–2) when you need a *hard guarantee* that the
requirements hold, and a **Shield Loss** (part 3) when you want a *soft nudge* towards them.